# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)

# Display dataset name and description
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the `mlcroissant` dataset object.

All references to entities use their Croissant schema `@id` for unambiguous referencing.

First, list available record sets, then inspect their fields and columns.

In [ ]:
# List all the record sets and associated fields (by @id)

record_sets = list(dataset.record_sets)
print('Available record sets:')
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs.id}")
    print(f"  Description: {rs.description}")
    print(f"  Fields (@id):")
    for field in rs.fields:
        print(f"    • {field.name} (@id: {field.id}, dataType: {field.data_type})")
    print('  Columns:')
    for col in rs.columns:
        print(f"    • {col.name} (@id: {col.id})")
    print('-' * 40)

# As an example, let's show 1-2 records for each record set
for rs in record_sets:
    print(f"\nSample records for RecordSet @id: {rs.id}, name: {rs.name}")
    try:
        for i, record in enumerate(dataset.records(record_set=rs.id)):
            print(record)
            if i >= 1:
                break
    except Exception as e:
        print(f"  Could not display records: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

We'll load all available record sets using their `@id`, and show the columns for a selected record set.

In [ ]:
# Extract all data into DataFrames (using @id for record sets)
# Update this section if you know the record set @ids you'd like to analyze

record_set_ids = [rs.id for rs in dataset.record_sets]
# Store DataFrames by record set @id
dataframes = {}

for record_set_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            dataframes[record_set_id] = pd.DataFrame(records)
            print(f"Loaded {len(records)} records for RecordSet {record_set_id}")
        else:
            print(f"No records found for RecordSet {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Select a record set with actual data for further exploration
example_record_set_id = None
for k, v in dataframes.items():
    if not v.empty:
        example_record_set_id = k
        break

if example_record_set_id:
    print(f"\nColumns in {example_record_set_id}:")
    print(dataframes[example_record_set_id].columns.tolist())
    print(f"\nSample rows:")
    display(dataframes[example_record_set_id].head())
else:
    print("No data available in any record set for DataFrame exploration.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We will select a numeric field, filter records, normalize the field, and group by a categorical field. All field and record set access is via `@id`.

Update the field and group ids as needed. You can refer to the above outputs to select the appropriate `@id` for numeric and group fields.

In [ ]:
# Example: Numeric field analysis and group statistics

import numpy as np

# Example values (replace with an actual numeric_field_id and group_field_id from overview)
numeric_field_id = None
group_field_id = None

# Try to auto-select a candidate numeric and group field
df = dataframes[example_record_set_id] if example_record_set_id else None

if df is not None and not df.empty:
    numeric_candidates = df.select_dtypes(include=np.number).columns.tolist()
    if numeric_candidates:
        numeric_field_id = numeric_candidates[0]
        print(f"Selected numeric field: {numeric_field_id}")
    group_candidates = df.select_dtypes(include='object').columns.tolist()
    if group_candidates:
        group_field_id = group_candidates[0]
        print(f"Selected group field: {group_field_id}")

    if numeric_field_id:
        # Filter by a threshold
        threshold = df[numeric_field_id].mean() if np.issubdtype(df[numeric_field_id].dtype, np.number) else 0
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by field
        if group_field_id and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
            print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
            display(grouped_df.head())
    else:
        print("No numeric field detected for EDA.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. Here, we show a histogram for the selected numeric field and a bar plot for group means.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().nlargest(10).reset_index()
        sns.barplot(x=numeric_field_id, y=group_field_id, data=grouped, orient='h')
        plt.title(f"Top 10 {group_field_id} group means for {numeric_field_id}")
        plt.xlabel(f"Mean of {numeric_field_id}")
        plt.ylabel(group_field_id)
        plt.tight_layout()
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion
We explored the FAIR^2 dataset, loaded via the Croissant schema using `mlcroissant`, listing its record sets, and demonstrating basic EDA and visualization referencing fields by their `@id`. For deeper analysis, consult the dataset's documentation, use specific `@id` for fields of interest, and tailor the notebook to your research questions.

For further analysis, see the [mlcroissant documentation](https://mlcommons.github.io/croissant/python/).